# Annotator export explorer

Interactive notebook for **reading**, **combining**, **filtering**, and **plotting** Block Annotator exports.

**Workflow**
1. Set `OUTPUT_FOLDER` and selection filters in the config cell.
2. Scan `*_annotations.json` files → inspect as tables.
3. Filter events (by block, type, animal, or custom pandas masks).
4. Load underlying **block objects** (`BlockDataCache`: eye CSVs, sync table, OE).
5. Extract aligned snippets and plot (same logic as the GUI / `annotator_plot_core.py`).

When you are happy with the logic here, copy the relevant cells into a standalone script.

**Requires:** `conda activate eye_annotator` and `pip install -e . --no-deps` from the repo root.

In [ ]:
# --- imports & path setup ---
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Repo root (adjust if you moved the notebook)
REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "eye_tracking_system_tools").exists():
    REPO_ROOT = Path.cwd().parent.parent  # when cwd is scripts/plots

PLOTS_DIR = REPO_ROOT / "scripts" / "plots"
if str(PLOTS_DIR) not in sys.path:
    sys.path.insert(0, str(PLOTS_DIR))

from annotator_plot_core import (
    PlotStyle,
    STREAM_EP,
    STREAM_L_DEG,
    STREAM_L_PUPIL,
    STREAM_R_DEG,
    STREAM_R_PUPIL,
    STREAM_LABELS,
    _style_axes,
    collect_snippets,
    events_to_dataframe,
    export_figures,
    load_block_cache_headless,
    records_from_dataframe,
    render_stream_axis,
    scan_annotations,
    summaries_to_dataframe,
    ylabel_for_normalization,
)
from eye_tracking_system_tools.annotation.event_explorer.catalog import (
    build_catalog,
    discover_annotation_files,
    load_events_from_file,
)

%matplotlib inline
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42

print(f"Repo root: {REPO_ROOT}")
print(f"Plots dir: {PLOTS_DIR}")

## 1. Configuration — edit this cell

All user-facing knobs live here. Re-run downstream cells after changes.

In [ ]:
# ===================== EDIT ME =====================
OUTPUT_FOLDER = REPO_ROOT / "_annotator_out"

# Limit to specific annotation files (filenames only), or None = all in folder
SELECTED_ANNOTATION_FILES: list[str] | None = None
# e.g. ["PV_106_2025_09_04_block_015_annotations.json"]

# Event types to keep (None = all types found)
SELECTED_EVENT_TYPES: list[str] | None = None
# e.g. ["saccade", "prey_capture"]

# Optional extra filter on the events table (pandas boolean Series or None)
# Applied after type/block selection — edit in section 3 if needed
CUSTOM_EVENT_MASK = None

# Streams to extract/plot
PLOT_L_PUPIL = True
PLOT_R_PUPIL = True
PLOT_L_DEG = False
PLOT_R_DEG = False
PLOT_EP = False
EP_CHANNELS = [1]  # HS channels when PLOT_EP is True

# Alignment window & normalization
HALF_WINDOW_MS = 100.0
NORMALIZATION = "within_block_zscore"  # none | within_block_zscore | per_trial_zscore
PLOT_MODE = "both"  # average | individual | both
N_GRID = 201

# Figure style
FIGSIZE = (2.4, 1.8)  # width, height per panel
DPI = 120  # use 300 for publication export
LABEL_FONTSIZE = 9.0
TICK_FONTSIZE = 8.0
MEAN_LINE_WIDTH = 1.8
TRIAL_ALPHA = 0.35

# PDF export (set paths to export; leave None to skip)
EXPORT_MAIN_PDF: Path | None = None
EXPORT_LEGEND_PDF: Path | None = None
# EXPORT_MAIN_PDF = OUTPUT_FOLDER / "notebook_export.pdf"
# EXPORT_LEGEND_PDF = OUTPUT_FOLDER / "notebook_export_legend.pdf"
# ===================================================

## 2. Discover annotation exports

Each row is one `*_annotations.json` (one annotated block).

In [ ]:
summaries = scan_annotations(OUTPUT_FOLDER)
blocks_df = summaries_to_dataframe(summaries)

if blocks_df.empty:
    raise FileNotFoundError(f"No *_annotations.json in {OUTPUT_FOLDER}")

display(blocks_df)
print(f"\n{len(blocks_df)} annotation file(s), {blocks_df['n_events'].sum()} total events")

## 3. Build & filter the event catalog

`catalog` is the full list of `EventRecord` objects.  
`events_df` is a flat table — filter with pandas, then map back with `records_from_dataframe`.

In [ ]:
json_paths = discover_annotation_files(scan_dirs=[OUTPUT_FOLDER], recursive=False)

if SELECTED_ANNOTATION_FILES:
    wanted = set(SELECTED_ANNOTATION_FILES)
    json_paths = [p for p in json_paths if p.name in wanted]

catalog = build_catalog(json_paths)
events_df = events_to_dataframe(catalog)

# --- standard filters ---
mask = pd.Series(True, index=events_df.index)
if SELECTED_EVENT_TYPES:
    mask &= events_df["event_type"].isin(SELECTED_EVENT_TYPES)
mask &= events_df["block_exists"]

# --- your custom filter (optional) ---
if CUSTOM_EVENT_MASK is not None:
    mask &= CUSTOM_EVENT_MASK

filtered_df = events_df.loc[mask].copy()
selected_records = records_from_dataframe(catalog, filtered_df)

print(f"Selected {len(selected_records)} / {len(catalog)} events")
display(filtered_df.sort_values(["animal_call", "block_num", "timepoint_ms"]))

### Filter examples (copy/edit)

```python
# By animal
mask = events_df["animal_call"] == "PV_106"

# By block number
mask = events_df["block_num"].isin(["015", "019"])

# By time range on block timeline (ms)
mask = events_df["timepoint_ms"].between(70_000, 80_000)

# By note text
mask = events_df["note"].str.len() > 0

filtered_df = events_df.loc[mask]
selected_records = records_from_dataframe(catalog, filtered_df)
```

## 4. Inspect raw JSON (optional)

Peek at the on-disk export schema for one file.

In [ ]:
import json

example_json = Path(json_paths[0])
raw = json.loads(example_json.read_text(encoding="utf-8"))

print("Top-level keys:", list(raw.keys()))
print("N events:", len(raw.get("events", [])))
print("block_path:", raw.get("block_path"))
print("\nFirst event:")
display(pd.json_normalize([raw["events"][0]]))

## 5. Access block objects

`BlockDataCache` holds everything needed to pull traces:
- `final_sync_df` — master sync table
- `le_df` / `re_df` — eye pipeline CSVs
- `column_map` — auto-detected pupil / degrees columns
- `oe_rec` — Open Ephys recording handle (for EP)

The cache is reused across events from the same block.

In [ ]:
class NotebookLog:
    """Print load messages in the notebook."""
    def info(self, msg: str) -> None:
        print(f"INFO  {msg}")
    def warn(self, msg: str) -> None:
        print(f"WARN  {msg}")

log = NotebookLog()
block_cache: dict[Path, object] = {}

# Pick one selected event (change index to explore others)
rec = selected_records[0]
cache = load_block_cache_headless(rec, log, block_cache)

print(f"Event: {rec.event_type} @ {rec.timepoint_ms:.1f} ms")
print(f"Block: {cache.block_path}")
print(f"Sample rate: {cache.sample_rate_hz} Hz")
print(f"Column map: pupil={cache.column_map.pupil}, L deg={cache.column_map.l_degrees}, R deg={cache.column_map.r_degrees}")
print(f"OE recording: {'yes' if cache.oe_rec is not None else 'no'}")
print(f"L eye CSV rows: {len(cache.le_df) if cache.le_df is not None else 0}")
print(f"R eye CSV rows: {len(cache.re_df) if cache.re_df is not None else 0}")

display(cache.final_sync_df.head(3))
if cache.le_df is not None:
    display(cache.le_df.head(3))

## 6. Extract snippets

Low-level access: one aligned trace per event × stream.  
Keys: `l_pupil`, `r_pupil`, `l_degrees`, `r_degrees`, `ep:1`, …

In [ ]:
stream_ids: list[str] = []
if PLOT_L_PUPIL:
    stream_ids.append(STREAM_L_PUPIL)
if PLOT_R_PUPIL:
    stream_ids.append(STREAM_R_PUPIL)
if PLOT_L_DEG:
    stream_ids.append(STREAM_L_DEG)
if PLOT_R_DEG:
    stream_ids.append(STREAM_R_DEG)
if PLOT_EP:
    stream_ids.append(STREAM_EP)

snippets_by_key = collect_snippets(
    selected_records,
    stream_ids,
    EP_CHANNELS,
    HALF_WINDOW_MS,
    NORMALIZATION,
    log,
)

for key, snips in snippets_by_key.items():
    print(f"{key}: {len(snips)} snippet(s)")

# Inspect one snippet
example_key = next(k for k, v in snippets_by_key.items() if v)
snip = snippets_by_key[example_key][0]
print(f"\nExample {example_key}: len={len(snip.values)}, source={snip.source}")
print(f"  time_rel_ms [{snip.time_rel_ms[0]:.1f}, {snip.time_rel_ms[-1]:.1f}]")
print(f"  meta: {snip.meta}")

fig, ax = plt.subplots(figsize=(4, 2))
ax.plot(snip.time_rel_ms, snip.values, lw=1)
ax.axvline(0, color="#aaa", ls=":")
ax.set_xlabel("Time from event (ms)")
ax.set_title(f"Single trial — {STREAM_LABELS.get(snip.stream_id, example_key)}")
plt.tight_layout()
plt.show()

## 7. Plot — average / individuals / both

Uses the same stacking & styling as the GUI. Re-run after editing the config cell.

In [ ]:
from annotator_plot_core import stack_stream_data

style = PlotStyle(
    figsize=FIGSIZE,
    dpi=DPI,
    label_fontsize=LABEL_FONTSIZE,
    tick_fontsize=TICK_FONTSIZE,
    mean_line_width=MEAN_LINE_WIDTH,
    trial_alpha=TRIAL_ALPHA,
)

selected_ids = {r.event_id for r in selected_records}
plot_series = []
for sid in stream_ids:
    if sid == STREAM_EP:
        for ch in EP_CHANNELS:
            key = f"ep:{ch}"
            snips = [s for s in snippets_by_key.get(key, []) if s.event_id in selected_ids]
            data = stack_stream_data(snips, key, N_GRID, HALF_WINDOW_MS)
            if data:
                plot_series.append(data)
    else:
        snips = [s for s in snippets_by_key.get(sid, []) if s.event_id in selected_ids]
        data = stack_stream_data(snips, sid, N_GRID, HALF_WINDOW_MS)
        if data:
            plot_series.append(data)

if not plot_series:
    raise RuntimeError("No plottable data — check filters and block paths.")

ylabel = ylabel_for_normalization(NORMALIZATION)
n_panels = len(plot_series)
fig_h = FIGSIZE[1] * n_panels if n_panels > 1 else FIGSIZE[1]
fig, axes = plt.subplots(n_panels, 1, figsize=(FIGSIZE[0], fig_h), sharex=True, squeeze=False)

for ax, data in zip(axes.ravel(), plot_series):
    render_stream_axis(ax, data, mode=PLOT_MODE, style=style)
    _style_axes(ax, style, ylabel)
    ax.set_title(f"{data.label}  (N={data.n_trials})", fontsize=TICK_FONTSIZE)
    ax.set_xlim(float(data.grid[0]), float(data.grid[-1]))

fig.subplots_adjust(hspace=0.3 if n_panels > 1 else 0.08)
plt.show()

## 8. Export PDFs (optional)

Set `EXPORT_MAIN_PDF` and `EXPORT_LEGEND_PDF` in the config cell, then run.

In [ ]:
if EXPORT_MAIN_PDF:
    export_style = PlotStyle(
        figsize=FIGSIZE,
        dpi=300,
        label_fontsize=LABEL_FONTSIZE,
        tick_fontsize=TICK_FONTSIZE,
        mean_line_width=MEAN_LINE_WIDTH,
        trial_alpha=TRIAL_ALPHA,
    )
    export_figures(
        plot_series,
        mode=PLOT_MODE,
        normalization=NORMALIZATION,
        style=export_style,
        main_path=EXPORT_MAIN_PDF,
        legend_path=EXPORT_LEGEND_PDF,
    )
    print(f"Saved {EXPORT_MAIN_PDF}")
    if EXPORT_LEGEND_PDF:
        print(f"Saved {EXPORT_LEGEND_PDF}")
else:
    print("Set EXPORT_MAIN_PDF in config to write PDFs.")

## 9. Your experiments

Use this cell (and new cells below) to prototype filters, custom normalizations, or per-animal plots.  
When stable, copy the logic into `scripts/plots/your_script.py`.

In [ ]:
# Example: group events by type and print counts
filtered_df.groupby(["animal_call", "block_num", "event_type"]).size().unstack(fill_value=0)

# Example: slice sync table around one event
# row = selected_records[0]
# c = load_block_cache_headless(row, log, block_cache)
# t0, t1 = row.start_ms, row.end_ms
# c.final_sync_df.query("@t0 <= ms_axis <= @t1")  # if ms_axis column exists